## ✅ ขั้นตอนที่ 1: ติดตั้งไลบรารีและโหลดเอกสาร

In [1]:
%pip install nest_asyncio

In [2]:
import nest_asyncio
nest_asyncio.apply()

In [3]:
# ติดตั้งไลบรารีที่จำเป็น
!pip install -q llama-index chromadb nest-asyncio python-docx
!pip install -q ngrok==1.3.0
!pip install -q uvicorn==0.29.0
!pip install -q fastapi==0.111.0
!pip install -q loguru==0.7.2
!pip install -q gdown
!pip install -q docx2txt
%pip install -Uq llama-index-embeddings-huggingface
%pip install -Uq chromadb
%pip install -Uq llama-index-vector-stores-chroma
%pip install -q llama-index-llms-openrouter

# สร้างโฟลเดอร์เก็บไฟล์
!mkdir -p ./data

file_id = "1NOzM7fEtOVGYQNj52Tp36GdSSzZS3U2h"
destination = "./data/1.pdf"

!gdown --id {file_id} -O {destination}


# ดาวน์โหลดเอกสารตัวอย่างสำหรับสร้าง index
!curl -L -o ./data/cv.docx "https://raw.githubusercontent.com/9meo/llm-training/refs/heads/main/material/cv_thai_sample.docx"
!curl -L -o ./data/llm_basics.txt "https://raw.githubusercontent.com/9meo/llm-training/refs/heads/main/material/llm_basics.txt"
!curl -L -o ./data/elephant.txt "https://raw.githubusercontent.com/9meo/llm-training/refs/heads/main/material/thai_elephant_knowledge.txt"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 92.1 MB/s eta 0:00:

## ✅ ขั้นตอนที่ 2: โหลดเอกสาร และสร้าง Embedding + Index

In [4]:
from llama_index.llms.openrouter import OpenRouter
from llama_index.core.llms import ChatMessage
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

# กำหนด LLM ที่จะใช้ (สามารถเปลี่ยน model ได้ตามต้องการ)
llm = OpenRouter(
   api_key="",  # 🔑 ใส่ OpenRouter API Key ที่นี่
    max_tokens=256,
    context_window=4096,
    #model="google/gemma-3-27b-it:free",
    model="meta-llama/llama-4-maverick:free",
)




In [5]:
message = ChatMessage(role="user", content="เล่นตลกให้ดูหน่อย อย่างสีเหลือง=เยลโล่  มะม่วง=แมงโก้ มีด=")
resp = llm.chat([message])
print(resp)

assistant: มีด = ไนฟ์ (Knife)


In [6]:
# โหลดเอกสารจากโฟลเดอร์ ./data
docs = SimpleDirectoryReader(input_dir="./data").load_data()

In [7]:
import pprint
pprint.pprint(docs)

[Document(id_='c0162304-8b2a-40a6-8619-765d7335ab38', embedding=None, metadata={'page_label': '1', 'file_name': '1.pdf', 'file_path': '/content/data/1.pdf', 'file_type': 'application/pdf', 'file_size': 3857399, 'creation_date': '2025-04-30', 'last_modified_date': '2025-03-19'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='ปัญญาประดิษฐ์เพื่อการประเมินการกัดกร่อน\nในบรรยากาศและแผนที่การกัดกร่อน:\nความยั่งยืนและความปลอดภัยของโครงสร้างเหล็กกล้า\nAI in Atmospheric Corrosion Assessment and Corroison Map:\nSteel Infrastructure Safety and Sustainability\n การกัดกร่อนของเหล็กกล้าโครงสร้างใน\nบรรยากาศขึ้นอยู่กับปัจจัยสภาพอากาศและ\nสภาพแวดล้อม เอ็

In [8]:
docs[0].__dict__

{'id_': 'c0162304-8b2a-40a6-8619-765d7335ab38',
 'embedding': None,
 'metadata': {'page_label': '1',
  'file_name': '1.pdf',
  'file_path': '/content/data/1.pdf',
  'file_type': 'application/pdf',
  'file_size': 3857399,
  'creation_date': '2025-04-30',
  'last_modified_date': '2025-03-19'},
 'excluded_embed_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'excluded_llm_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'relationships': {},
 'metadata_template': '{key}: {value}',
 'metadata_separator': '\n',
 'text_resource': MediaResource(embeddings=None, data=None, text='ปัญญาประดิษฐ์เพื่อการประเมินการกัดกร่อน\nในบรรยากาศและแผนที่การกัดกร่อน:\nความยั่งยืนและความปลอดภัยของโครงสร้างเหล็กกล้า\nAI in Atmospheric Corrosion Assessment and Corroison Map:\nSteel Infrastructure Safety and Sustainability\n การกัดกร่อนของเหล็กกล้าโครงสร้าง

In [9]:
docs[1].__dict__

{'id_': '299cf07b-08fa-4d80-b7cf-bca7a3c9af8c',
 'embedding': None,
 'metadata': {'file_name': 'cv.docx',
  'file_path': '/content/data/cv.docx',
  'file_type': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
  'file_size': 37428,
  'creation_date': '2025-04-30',
  'last_modified_date': '2025-04-30'},
 'excluded_embed_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'excluded_llm_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'relationships': {},
 'metadata_template': '{key}: {value}',
 'metadata_separator': '\n',
 'text_resource': MediaResource(embeddings=None, data=None, text='ประวัติส่วนตัว\n\nชื่อ: นายสมชาย ใจดี\n\nวันเกิด: 15 มกราคม 2533\n\nที่อยู่: 123 ถนนสุขุมวิท, เขตคลองเตย, กรุงเทพมหานคร 10110\n\nโทรศัพท์: 081-234-5678\n\nอีเมล: somchai.jai@gmail.com\n\nประวัติการศึกษา\n\nมหาวิทยาลัยธรรมศาสต

In [10]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

# ตั้งค่าให้แยกเอกสารเป็น chunk ย่อย ๆ
text_splitter = SentenceSplitter(
    separator=" ",       # ใช้เว้นวรรคเป็นตัวแยก
    chunk_size=1024,     # ขนาดของแต่ละ chunk (จำนวนตัวอักษร)
    chunk_overlap=128    # ความซ้อนของข้อความระหว่างแต่ละ chunk
)

# สร้าง ingestion pipeline ที่ประกอบด้วย text_splitter
pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
    ]
)

# รัน pipeline กับเอกสารที่โหลดไว้ เพื่อแยกเป็น chunks
nodes = pipeline.run(
    documents=docs,
    in_place=True,       # แก้ไขเอกสารเดิมแทนที่จะสร้างใหม่
    show_progress=True,  # แสดง progress bar ขณะรัน
)


Parsing nodes:   0%|          | 0/4 [00:00<?, ?it/s]

In [11]:
nodes[2].__dict__

{'id_': '91cd3986-1654-401e-b472-c6ccd5290fea',
 'embedding': None,
 'metadata': {'page_label': '1',
  'file_name': '1.pdf',
  'file_path': '/content/data/1.pdf',
  'file_type': 'application/pdf',
  'file_size': 3857399,
  'creation_date': '2025-04-30',
  'last_modified_date': '2025-03-19'},
 'excluded_embed_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'excluded_llm_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'relationships': {<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='c0162304-8b2a-40a6-8619-765d7335ab38', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page_label': '1', 'file_name': '1.pdf', 'file_path': '/content/data/1.pdf', 'file_type': 'application/pdf', 'file_size': 3857399, 'creation_date': '2025-04-30', 'last_modified_date': '2025-03-19'}, hash='8e83601cfc7c9fc334a6af04035e1eae07e3e8fdc3d4d344198

In [12]:
# ใช้โมเดล HuggingFace สำหรับสร้าง embedding
hf_embeddings = HuggingFaceEmbedding(model_name="BAAI/bge-m3")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [13]:
# เริ่มต้น client ของ ChromaDB และสร้าง collection สำหรับเก็บเวกเตอร์
db = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = db.get_or_create_collection("rag_demo")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# สร้าง storage context สำหรับบันทึกข้อมูล
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# สร้าง index จากเอกสาร โดยใช้ embedding ที่เรากำหนด
index = VectorStoreIndex(
    nodes=nodes, storage_context=storage_context, embed_model=hf_embeddings,show_progress=True,  # แสดง progress bar ขณะรัน
)

Generating embeddings:   0%|          | 0/9 [00:00<?, ?it/s]

In [14]:
# สร้าง query engine สำหรับถามคำถามจาก index
query_engine = index.as_query_engine(llm=llm)

In [15]:
# ตั้งคำถามที่ต้องการ
question = "ช้างไทยมีลักษณะอย่างไร"

# ใช้ query_engine ในการค้นหาคำตอบจากข้อมูลใน index
response = query_engine.query(question)

# แสดงผลลัพธ์
print("คำถาม:", question)
print("คำตอบ:", response)


คำถาม: ช้างไทยมีลักษณะอย่างไร
คำตอบ: ช้างไทยมีลักษณะทางกายภาพ เช่น ความสูงประมาณ 2.5 - 3.2 เมตร น้ำหนัก 2,700 - 4,500 กิโลกรัม และอายุขัย 60 - 80 ปี นอกจากนี้ยังมีลักษณะเฉพาะ เช่น หัวเป็นโค้ง คอหนา และหูขนาดเล็กกว่าช้างแอฟริกา


In [16]:
# แสดงแหล่งอ้างอิง (เช่น ชื่อไฟล์ หรือเนื้อหาบางส่วนที่เกี่ยวข้อง)
for node in response.source_nodes:
    print("--- แหล่งข้อมูลที่ใช้ ---")
    print(node.node.text[:300])  # แสดงเฉพาะ 300 ตัวอักษรแรก


--- แหล่งข้อมูลที่ใช้ ---
## ความรู้เกี่ยวกับช้างไทย

### 1. ช้างไทยคืออะไร?
ช้างไทย (Asian Elephant - Elephas maximus) เป็นสัตว์สัญลักษณ์ของประเทศไทยและมีบทบาทสำคัญทั้งในวัฒนธรรม ประวัติศาสตร์ และเศรษฐกิจของประเทศ โดยช้างไทยจัดอยู่ในกลุ่มช้างเอเชีย ซึ่งมีขนาดเล็กกว่าช้างแอฟริกา และมีลักษณะเฉพาะที่แตกต่างกัน เช่น หัวเป็นโค้ง
--- แหล่งข้อมูลที่ใช้ ---
การอนุรักษ์ช้างไทย
องค์กรและหน่วยงานหลายแห่งกำลังดำเนินมาตรการเพื่อปกป้องและอนุรักษ์ช้างไทย เช่น:
- การส่งเสริมศูนย์อนุรักษ์ช้าง เช่น ศูนย์อนุรักษ์ช้างไทยที่จังหวัดลำปาง
- การพัฒนาแนวทางการท่องเที่ยวเชิงอนุรักษ์ที่เป็นมิตรกับช้าง
- การออกกฎหมายเพื่อปกป้องช้างป่าและช้างบ้าน เช่น พ.ร.บ. สงวนและคุ้มครอ


## ✅ ขั้นตอนที่ 3: สร้าง FastAPI endpoint สำหรับทำการค้นหา

In [17]:
NGROK_AUTH_TOKEN = "88CcPMFj8rK6xJDqvNRHp_6DJ8J4BPGB7nfpa1aykmC"
APPLICATION_PORT = 5000
NGROK_EDGE = "edge:edghts_"

In [18]:
!pip install -q pyngrok

In [26]:
import os
import nest_asyncio
import threading
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok

# 🔐 ใส่ ngrok token ตรงนี้ (จาก https://dashboard.ngrok.com)
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# สร้าง FastAPI
app = FastAPI()

class QueryRequest(BaseModel):
    query: str

@app.post("/search")
def search(request: QueryRequest):
    response = query_engine.query(request.query)
    return {"query": request.query, "response": str(response)}

# รัน FastAPI บน Colab
nest_asyncio.apply()

def run():
    import uvicorn
    from uvicorn import Config, Server
    config = Config(app=app, host="0.0.0.0", port=5000, log_level="info")
    server = Server(config)

    import asyncio
    loop = asyncio.get_event_loop()
    loop.run_until_complete(server.serve())

# รันใน thread
threading.Thread(target=run).start()

# เปิด ngrok
public_url = ngrok.connect(5000)
print(f"🔗 FastAPI endpoint: {public_url}/search")


INFO:     Started server process [241]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 5000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: Your account may not run more than 3 tunnels over a single ngrok agent session.\nThe tunnels already running on this session are:\ntn_2wRUGpbT1hgUnhMdiX3HjYrcxKe, tn_2wRUKWMoV79bfs3Z4nMaiMMUlxU, tn_2wRUNPSGXlRSa539nUkXTu1kpjn\n\r\n\r\nERR_NGROK_324\r\n"}}


## ✅ วิธีทดลองใช้งาน API

In [ ]:
# ตัวอย่างคำสั่ง curl สำหรับเรียกใช้งาน API ที่ได้จาก ngrok
# อย่าลืมแทนที่ URL ด้านล่างด้วยลิงก์ที่ได้จาก ngrok

!curl -X POST https://87d3-35-231-44-44.ngrok-free.app/search \
  -H "Content-Type: application/json" \
  -d '{"query": "ช้างไทยกินอะไร"}'


INFO:     35.231.44.44:0 - "POST /search HTTP/1.1" 200 OK
{"query":"ช้างไทยกินอะไร","response":"\nช้างไทยกินพืช หญ้า ใบไม้ ผลไม้ และเปลือกไม้ ได้มากถึง 150 กิโลกรัมต่อวัน"}